In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, roc_curve
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [8]:
JM1_CSV_URL = "https://raw.githubusercontent.com/ApoorvaKrisna/NASA-promise-dataset-repository/main/jm1.csv"

df = pd.read_csv(JM1_CSV_URL)
print(df.shape)
df.head()

(13204, 22)


,loc,v(g),ev(g),iv(g),n,v,l,d,i,e,...,lOCode,lOComment,lOBlank,locCodeAndComment,uniq_Op,uniq_Opnd,total_Op,total_Opnd,branchCount,defects
0,1.1,1.4,1.4,1.4,1.3,1.30,1.30,1.30,1.30,1.30,...,2,2,2,2,1.2,1.2,1.2,1.2,1.4,False
1,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.00,1.00,...,1,1,1,1,1.0,1.0,1.0,1.0,1.0,True
2,72.0,7.0,1.0,6.0,198.0,1134.13,0.05,20.31,55.85,23029.10,...,51,10,8,1,17.0,36.0,112.0,86.0,13.0,True
3,190.0,3.0,1.0,3.0,600.0,4348.76,0.06,17.06,254.87,74202.67,...,129,29,28,2,17.0,135.0,329.0,271.0,5.0,True
4,37.0,4.0,1.0,4.0,126.0,599.12,0.06,17.19,34.86,10297.30,...,28,1,6,0,11.0,16.0,76.0,50.0,7.0,True


In [9]:
# Common target name in these mirrors:
possible_targets = [c for c in df.columns if c.lower() in ["defects", "defect", "bug", "is_defective", "defective"]]
print("Possible target columns:", possible_targets)

TARGET = possible_targets[0] if possible_targets else df.columns[-1]  # fallback
print("Using TARGET =", TARGET)

# Normalize target to 0/1
y_raw = df[TARGET]

# If it's strings like 'true/false' or 'yes/no'
if y_raw.dtype == "object":
    y = y_raw.astype(str).str.lower().map({"true":1,"false":0,"yes":1,"no":0,"y":1,"n":0,"1":1,"0":0})
    if y.isna().any():
        # try last resort: treat non-zero / non-empty as defective
        y = y_raw.astype(str).str.strip().ne("0").astype(int)
else:
    y = (pd.to_numeric(y_raw, errors="coerce").fillna(0) > 0).astype(int)

X = df.drop(columns=[TARGET])

print("Class balance:\n", y.value_counts(normalize=True).rename("ratio"))

Possible target columns: ['defects']
Using TARGET = defects
Class balance:
 defects
0    0.84073
1    0.15927
Name: ratio, dtype: float64


In [10]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
# Keep only numeric columns (JM1 is usually all numeric features)
X = X[numeric_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols)
    ],
    remainder="drop"
)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
lr = Pipeline(steps=[
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=None))
])

lr.fit(X_train, y_train)

lr_proba = lr.predict_proba(X_test)[:, 1]
lr_pred= (lr_proba >= 0.3).astype(int)


print("Logistic Regression")
print(confusion_matrix(y_test, lr_pred))
print(classification_report(y_test, lr_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, lr_proba))
print("PR-AUC:", average_precision_score(y_test, lr_proba))
fpr, tpr, thresholds = roc_curve(y_test, lr_proba)
optimal_idx = (tpr - fpr).argmax()
optimal_threshold = thresholds[optimal_idx]
print("Best threshold: ",optimal_threshold)

Logistic Regression
[[ 438 1782]
 [  22  399]]
              precision    recall  f1-score   support

           0     0.9522    0.1973    0.3269      2220
           1     0.1829    0.9477    0.3067       421

    accuracy                         0.3169      2641
   macro avg     0.5676    0.5725    0.3168      2641
weighted avg     0.8296    0.3169    0.3236      2641

ROC-AUC: 0.7212171791744239
PR-AUC: 0.36210627597127193
Best threshold:  0.48683073195755544


In [14]:
rf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)

rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred= (rf_proba >= 0.3).astype(int)

print("Random Forest")
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, rf_proba))
print("PR-AUC:", average_precision_score(y_test, rf_proba))
fpr, tpr, thresholds = roc_curve(y_test, rf_proba)
optimal_idx = (tpr - fpr).argmax()
optimal_threshold = thresholds[optimal_idx]
print("Best threshold: ",optimal_threshold)

Random Forest
[[1948  272]
 [ 240  181]]
              precision    recall  f1-score   support

           0     0.8903    0.8775    0.8838      2220
           1     0.3996    0.4299    0.4142       421

    accuracy                         0.8061      2641
   macro avg     0.6449    0.6537    0.6490      2641
weighted avg     0.8121    0.8061    0.8090      2641

ROC-AUC: 0.7873215852432004
PR-AUC: 0.4322081586582644
Best threshold:  0.175


In [15]:
# Get feature names
feature_names = numeric_cols
importances = rf.named_steps["model"].feature_importances_

fi = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
fi.head(15)

,feature,importance
0,loc,0.134663
8,i,0.060397
14,lOBlank,0.057373
5,v,0.057218
12,lOCode,0.054793
9,e,0.052673
11,t,0.051930
3,iv(g),0.050505
7,d,0.050214
20,branchCount,0.048566
